# 第 03 章：RAG 2.0 增强型数据检索

搭建外部图书馆，让通用智能体读上你的专属密卷！

## 1. 准备本地极简材料堆

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# 我们先在同层目录下直接创建一个公司规则文档
with open("knowledge.txt", "w", encoding="utf-8") as f:
    f.write("1001-AABBCC: 这是一张购买于 2026 年的主板，核心组件为 Z170X，保修期五年。\n")
    f.write("今天公司发布了新规，要求员工早上9:30之前全部打卡完毕，迟到扣减工资100元。\n")
    f.write("若遇到系统卡顿问题，请务必尝试长按电源键 5 秒强制重启，这是硬件级的保险措施。\n")
print("知识底座原始材料生成完成！")

知识底座原始材料生成完成！


## 2. 装机：切分与模型坐标仪翻译 (Loader -> Split -> Embed)

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

loader = TextLoader("./knowledge.txt", encoding="utf-8")
docs = loader.load()

# 开始连断裁刀机制
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
splits = text_splitter.split_documents(docs)
print(f"大文段已切碎，输出块堆总数: {len(splits)} 块。")

大文段已切碎，输出块堆总数: 2 块。


In [8]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-v4",
    api_key=os.getenv("BAILIAN_API_KEY"),
    base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
    check_embedding_ctx_length=False
)
print("向量化模型准备完毕")

向量化模型准备完毕


## 3. 管家进场，增量指纹核验入库 (Indexing API)
**可直接多次重复执行（Ctrl+Enter）下面这一段块！**体会 `SQLRecordManager` 作为大管家的强大把控度。

In [9]:
from langchain_chroma import Chroma
from langchain_classic.indexes import index, SQLRecordManager

vectorstore = Chroma(
    collection_name="my_private_knowledge",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

record_manager = SQLRecordManager("chroma/my_private_knowledge", db_url="sqlite:///record_manager_cache.sql")
record_manager.create_schema()

# 让管家清点后载入库（你会看到除了第一次是 added 外，其他全为 skipped！）
result = index(
    docs_source=splits,
    record_manager=record_manager,
    vector_store=vectorstore,
    cleanup="incremental",
    source_id_key="source"
)

print("入库管家盘点记录单:", result)

入库管家盘点记录单: {'num_added': 2, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}


In [10]:
from langchain_chroma import Chroma
from langchain_classic.indexes import index, SQLRecordManager

vectorstore = Chroma(
    collection_name="my_private_knowledge",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

record_manager = SQLRecordManager("chroma/my_private_knowledge", db_url="sqlite:///record_manager_cache.sql")
record_manager.create_schema()

# 让管家清点后载入库（你会看到除了第一次是 added 外，其他全为 skipped！）
result = index(
    docs_source=splits,
    record_manager=record_manager,
    vector_store=vectorstore,
    cleanup="incremental",
    source_id_key="source"
)

print("入库管家盘点记录单:", result)

入库管家盘点记录单: {'num_added': 0, 'num_updated': 0, 'num_skipped': 2, 'num_deleted': 0}


## 4. 组成最强裁判所 (RRF & Ensemble Retriever)
合并传统关键词打分与高维语义搜索两路偏向优势集。

In [13]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# 搭建小学究字眼搜索器
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 2

# 抽出玄幻老中医之寻意意搜器
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 组成裁判法官席
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5] 
)

query = "我的编号是 1001-AABBCC，请问是什么物件体系的？"
docs = hybrid_retriever.invoke(query)

print(f"【问题】：{query}\n")
print(f"【混合提取到的关键线索】：{docs[0].page_content.strip()}")

【问题】：我的编号是 1001-AABBCC，请问是什么物件体系的？

【混合提取到的关键线索】：若遇到系统卡顿问题，请务必尝试长按电源键 5 秒强制重启，这是硬件级的保险措施。


In [14]:
# 诊断：分别看看两位搜索专家的结果
docs_bm25 = bm25_retriever.invoke(query)
docs_vec = vector_retriever.invoke(query)

print(f"--- 小学究（BM25）搜到了什么？---")
print(docs_bm25[0].page_content if docs_bm25 else "啥也没搜到")

print(f"\n--- 老中医（Vector）搜到了什么？---")
print(docs_vec[0].page_content if docs_vec else "啥也没搜到")

--- 小学究（BM25）搜到了什么？---
若遇到系统卡顿问题，请务必尝试长按电源键 5 秒强制重启，这是硬件级的保险措施。

--- 老中医（Vector）搜到了什么？---
1001-AABBCC: 这是一张购买于 2026 年的主板，核心组件为 Z170X，保修期五年。
今天公司发布了新规，要求员工早上9:30之前全部打卡完毕，迟到扣减工资100元。


In [16]:
import jieba
from langchain_community.retrievers import BM25Retriever

# 1. 强制分词
def chinese_tokenizer(text):
    return list(jieba.cut(text))

# 2. 重新初始化
bm25_retriever = BM25Retriever.from_documents(
    splits, 
    preprocess_func=chinese_tokenizer # 这样它才能精准识别出 1001-AABBCC
)

/Users/cyrus/work/my/langchain-learn/.venv/lib/python3.12/site-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/Users/cyrus/work/my/langchain-learn/.venv/lib/python3.12/site-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/Users/cyrus/work/my/langchain-learn/.venv/lib/python3.12/site-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/1m/r6x_rj6x3d5_wxnlxvds6bvc0000gn/T/jieba.cache
Loading model cost 0.286 seconds.
Prefix dict has been built successfully.


In [17]:
# 抽出玄幻老中医之寻意意搜器
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 组成裁判法官席
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5] 
)

query = "我的编号是 1001-AABBCC，请问是什么物件体系的？"
docs = hybrid_retriever.invoke(query)

print(f"【问题】：{query}\n")
print(f"【混合提取到的关键线索】：{docs[0].page_content.strip()}")

【问题】：我的编号是 1001-AABBCC，请问是什么物件体系的？

【混合提取到的关键线索】：1001-AABBCC: 这是一张购买于 2026 年的主板，核心组件为 Z170X，保修期五年。
今天公司发布了新规，要求员工早上9:30之前全部打卡完毕，迟到扣减工资100元。


In [21]:
# 加入大模型
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """你是一个专业的知识库助手。
请根据提供的【线索】来回答用户的【问题】。
注意：
1. 仅提取与问题相关的线索。
2. 如果线索中包含无关信息（如打卡、天气等），请直接忽略。
3. 如果线索中没有答案，请礼貌地说不知道。"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "【问题】：{question}\n\n【线索】：{context}")
])

In [19]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 统一接入入口
llm = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("https://api.deepseek.com")
)
print("模型实例建立完成。")

模型实例建立完成。


In [26]:
from langchain_core.output_parsers import StrOutputParser

rag_chain = prompt | llm | StrOutputParser()

# 4. 运行！
# 我们把之前 hybrid_retriever 搜出来的碎片塞进去
context_text = "\n".join([doc.page_content for doc in docs]) # 融合多个碎片的文本

print("--召回--")
print(context_text)
print("----")

--召回--
1001-AABBCC: 这是一张购买于 2026 年的主板，核心组件为 Z170X，保修期五年。
今天公司发布了新规，要求员工早上9:30之前全部打卡完毕，迟到扣减工资100元。
若遇到系统卡顿问题，请务必尝试长按电源键 5 秒强制重启，这是硬件级的保险措施。
----


In [24]:
response = rag_chain.invoke({
    "question": query,
    "context": context_text
})

print(f"--- 最终提纯后的回答 ---\n{response}")

--- 最终提纯后的回答 ---
根据线索，您的编号 1001-AABBCC 对应的是主板物件体系。
